In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import time
import tracemalloc
import os

# Define the model classes (same as before)
class RNNCausalModel(nn.Module):
    def __init__(self, covariate_dim, treatment_dim, hidden_dim, output_dim, num_layers=2, dropout_prob=0.0):
        super(RNNCausalModel, self).__init__()
        self.covariate_dim = covariate_dim
        self.treatment_dim = treatment_dim
        self.hidden_dim = hidden_dim
        
        # RNN to process the sequence
        self.rnn = nn.GRU(
            input_size=covariate_dim + treatment_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_prob if num_layers > 1 else 0
        )
        
        # Prediction network
        self.output_net = nn.Sequential(
            nn.Linear(hidden_dim + covariate_dim + 3 * treatment_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, output_dim)
        )
        
        # Network for intermediate predictions
        self.covariate_pred = nn.Sequential(
            nn.Linear(hidden_dim + treatment_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, covariate_dim)
        )
    def forward(self, X_seq, A_seq, predict_sequence=False):
        batch_size = X_seq.size(0)
        
        # Ensure treatments have correct dimensions [batch, seq_len, treatment_dim]
        if A_seq.dim() == 2:  # If shape is [batch, 3]
            A_seq = A_seq.unsqueeze(-1)  # Shape becomes [batch, 3, 1]
        elif A_seq.dim() == 3 and A_seq.size(-1) != self.treatment_dim:
            A_seq = A_seq.view(batch_size, -1, self.treatment_dim)
        
        # Combine covariates and treatments
        rnn_input = torch.cat([X_seq, A_seq], dim=-1)  # Shape: [batch, 3, covariate_dim + treatment_dim]
        
        # Initialize hidden state
        h0 = torch.zeros(self.rnn.num_layers, batch_size, self.hidden_dim).to(X_seq.device)
        
        # RNN forward pass
        rnn_out, _ = self.rnn(rnn_input, h0)  # rnn_out: [batch, 3, hidden_dim]
        
        # Final prediction components
        final_hidden = rnn_out[:, -1, :]  # Last time step
        X0 = X_seq[:, 0, :]  # Baseline covariates
        A_all = A_seq.view(batch_size, -1)  # Flatten treatments [batch, 3*treatment_dim]
        
        # Final prediction
        y_pred = self.output_net(torch.cat([final_hidden, X0, A_all], dim=-1))
        
        if predict_sequence:
            # Predict intermediate covariates
            X_preds = []
            for t in range(A_seq.size(1)):
                X_next = self.covariate_pred(torch.cat([rnn_out[:, t, :], A_seq[:, t, :]], dim=-1))
                X_preds.append(X_next)
            return y_pred, torch.stack(X_preds, dim=1)
        
        return y_pred

def train_rnn_model(model, train_loader, val_loader, num_epochs=50, learning_rate=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    
    criterion = nn.MSELoss(reduction='none')
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        
        for X_batch, A_batch, weights_batch, y_batch in train_loader:
            # Move data to device
            X_batch = X_batch.to(device)
            A_batch = A_batch.to(device)
            weights_batch = weights_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()

            # Forward pass
            y_pred, X_preds = model(X_batch, A_batch, predict_sequence=True)
            
            # Compute losses
            loss_y = criterion(y_pred, y_batch)
            
            # Compare ALL predicted covariates (X_preds) with ALL actual covariates (X_batch)
            loss_X = criterion(X_preds, X_batch)  # Now comparing [batch, 3, 4] with [batch, 3, 4]
            # Option 1: Sum over both time and covariate dimensions (returns [500])
            loss_X_sum = loss_X.sum(dim=[1, 2])  # Shape: [500]

            # Option 2: Average over both dimensions (returns [500])
            loss_X_mean = loss_X.mean(dim=[1, 2])  # Shape: [500]

            # If you specifically need [500, 1], add an extra dimension
            loss_X_final = loss_X.sum(dim=[1, 2]).unsqueeze(1)  # Shape: [500, 1]
            # Weighted total loss
            total_loss = torch.mean(weights_batch * (loss_y + 0.5 * loss_X_final))
            total_loss.backward()
            optimizer.step()
            
            train_loss += total_loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_batch, A_batch, weights_batch, y_batch in val_loader:
                X_batch, A_batch = X_batch.to(device), A_batch.to(device)
                weights_batch, y_batch = weights_batch.to(device), y_batch.to(device)
                
                y_pred = model(X_batch, A_batch)
                val_loss += torch.mean(weights_batch * criterion(y_pred, y_batch)).item()
        
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}")
    
    return model


def estimate_rnn_causal_effect(model, X_val, A_val, treatment_pattern=[1,1,1]):
    """Estimate the average causal effect of Y(0,0,1) - Y(0,0,0)"""
    model.eval()
    device = next(model.parameters()).device
    
    # Create counterfactual treatment vectors with proper dimensions
    A_treatment = torch.tensor(treatment_pattern, dtype=torch.float32).view(1, 3, 1).to(device)
    A_control = torch.zeros(1, 3, 1).to(device)  # Shape: [1, 3, 1]
    
    y_treatment_list = []
    y_control_list = []
    
    with torch.no_grad():
        for X_sample in X_val:
            X_sample = X_sample.unsqueeze(0).to(device)  # Shape: [1, 3, 4]
            
            # Predict for both treatment scenarios
            y_treatment = model(X_sample, A_treatment)
            y_control = model(X_sample, A_control)
            
            y_treatment_list.append(y_treatment.item())
            y_control_list.append(y_control.item())
    
    # Compute average causal effect
    y_treatment = np.array(y_treatment_list)
    y_control = np.array(y_control_list)
    causal_effect = y_treatment - y_control
    average_causal_effect = np.mean(causal_effect)
    
    return average_causal_effect

from sklearn.linear_model import LogisticRegression

def process_single_file(file_path):
    """Process a single data file with logistic regression-based weights"""
    # Load and preprocess data
    data = pd.read_csv(file_path)
    y = data["Y_glomerular_filtration"].values
    treatments = data[["treatment_1", "treatment_2", "treatment_3"]].values
    covariates = data.drop(columns=["Y_glomerular_filtration", "treatment_1", "treatment_2", "treatment_3", 
                                  "ps_treatment_1", "ps_treatment_2", "ps_treatment_3"]).values
    covariates = covariates.reshape(-1, 3, 4)  # Shape: (n_samples, 3, 4)
    
    # Split data
    X_train, X_val, y_train, y_val, A_train, A_val, _, _ = train_test_split(
        covariates, y, treatments, np.zeros_like(treatments), test_size=0.05, random_state=42)
    
    # Convert to numpy arrays
    X_train_np = X_train.numpy() if torch.is_tensor(X_train) else X_train
    A_train_np = A_train.numpy() if torch.is_tensor(A_train) else A_train
    X_val_np = X_val.numpy() if torch.is_tensor(X_val) else X_val
    A_val_np = A_val.numpy() if torch.is_tensor(A_val) else A_val
    
    # Calculate marginal probabilities
    p_A1 = A_train_np[:, 0].mean()
    p_A2 = A_train_np[:, 1].mean()
    p_A3 = A_train_np[:, 2].mean()
    
    # Fit logistic models for each time point
    def get_weights(X, A, p_marginal):
        weights = []
        
        # Time point 1: P(A1|X1)
        X1 = X[:, 0, :]
        model1 = LogisticRegression().fit(X1, A[:, 0])
        pred1 = model1.predict_proba(X1)[:, 1]
        w1 = np.where(A[:, 0] == 1, 1/pred1, 1/(1-pred1))
        weights.append(w1)
        
        # Time point 2: P(A2|X1,A1,X2)
        X2_features = np.concatenate([X[:, 0, :], A[:, 0:1], X[:, 1, :]], axis=1)
        model2 = LogisticRegression().fit(X2_features, A[:, 1])
        pred2 = model2.predict_proba(X2_features)[:, 1]
        w2 = np.where(A[:, 1] == 1, 1/pred2, 1/(1-pred2))
        weights.append(w2)
        
        # Time point 3: P(A3|X1,A1,X2,A2,X3)
        X3_features = np.concatenate([X[:, 0, :], A[:, 0:1], X[:, 1, :], 
                                    A[:, 1:2], X[:, 2, :]], axis=1)
        model3 = LogisticRegression().fit(X3_features, A[:, 2])
        pred3 = model3.predict_proba(X3_features)[:, 1]
        w3 = np.where(A[:, 2] == 1, 1/pred3, 1/(1-pred3))
        weights.append(w3)
        
        return np.prod(weights, axis=0)
    def compute_weights(A, ps):
        """
        Calculate weights with exponential decay from 0.5
    
        Args:
            A: Treatment assignments tensor (shape: [batch_size, 3])
            ps: Propensity scores tensor (shape: [batch_size, 3])
    
        Returns:
            weights: Combined weights tensor (shape: [batch_size])
        """
        weights = []
        for i in range(3):
            # Get treatment and propensity for this time point
            a = A[:, i]
            p = ps[:, i]
        
            # Calculate symmetric distance from 0.5
            distance = torch.abs(p - 0.5)
        
            # Exponential decay weight (max at p=0.5, decays symmetrically)
            # Using exp(-10*distance^2) for smooth decay
            base_weight = torch.exp(-10 * (distance ** 2))
        
            # Apply treatment-specific weighting
            w = torch.where(a == 1, base_weight / p, base_weight / (1 - p))
        
            weights.append(w)
    
        return torch.stack(weights, dim=1).prod(dim=1)
    # Calculate weights
    combined_weight_train = get_weights(X_train_np, A_train_np, [p_A1, p_A2, p_A3])
    combined_weight_val = get_weights(X_val_np, A_val_np, [p_A1, p_A2, p_A3])
    
    # Convert to tensors
    X_train = torch.tensor(X_train_np, dtype=torch.float32)
    X_val = torch.tensor(X_val_np, dtype=torch.float32)
    A_train = torch.tensor(A_train_np, dtype=torch.float32)
    A_val = torch.tensor(A_val_np, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    y_val = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
    combined_weight_train = torch.tensor(combined_weight_train, dtype=torch.float32)
    combined_weight_val = torch.tensor(combined_weight_val, dtype=torch.float32)
    
    # Create DataLoaders and train model (same as before)
    train_dataset = TensorDataset(X_train, A_train, combined_weight_train, y_train)
    val_dataset = TensorDataset(X_val, A_val, combined_weight_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
    
    model = RNNCausalModel(
        covariate_dim=4,
        treatment_dim=1,
        hidden_dim=32,
        output_dim=1,
        num_layers=2
    )
    
    tracemalloc.start()
    start_time = time.time()
    model = train_rnn_model(model, train_loader, val_loader)
    end_time = time.time()
    peak_memory = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    
    # Calculate causal effects
    effect_111 = estimate_rnn_causal_effect(model, X_train, A_train, [1,1,1])
    effect_011 = estimate_rnn_causal_effect(model, X_train, A_train, [0,1,1])
    
    return {
        'file_name': os.path.basename(file_path),
        'effect_111': effect_111,
        'effect_011': effect_011,
        'training_time': end_time - start_time,
        'peak_memory': peak_memory / 1024**2
    }

def process_directory(directory_path, output_csv):
    """Process all CSV files in directory and save results"""
    results = []
    processed_files = set()
    
    # Check if output file exists to resume processing
    if os.path.exists(output_csv):
        existing_results = pd.read_csv(output_csv)
        processed_files = set(existing_results['file_name'].tolist())
        results = existing_results.to_dict('records')
    
    # Get all CSV files in directory
    files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]
    
    for i, filename in enumerate(files):
        if filename in processed_files:
            print(f"Skipping already processed file: {filename}")
            continue
            
        file_path = os.path.join(directory_path, filename)
        print(f"Processing file {i+1}/{len(files)}: {filename}")
        
        try:
            result = process_single_file(file_path)
            results.append(result)
            
            # Save after each file in case of interruption
            pd.DataFrame(results).to_csv(output_csv, index=False)
            print(f"Saved results for {filename}")
        except Exception as e:
            print(f"Error processing {filename}: {str(e)}")
            continue
    
    return pd.DataFrame(results)



In [ ]:
# Main execution
if __name__ == "__main__":
    data_directory = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_07_sd_eps_3"
    output_file = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_07_sd_eps_3_results_RNN_modified.csv"
    
    print(f"Starting processing of directory: {data_directory}")
    results_df = process_directory(data_directory, output_file)
    print(f"Processing complete. Results saved to {output_file}")
    
    # Print summary statistics
    print("\nSummary Statistics:")
    print(f"Average Effect [1,1,1]: {results_df['effect_111'].mean():.4f}")
    print(f"Average Effect [0,1,1]: {results_df['effect_011'].mean():.4f}")
    print(f"Average Training Time: {results_df['training_time'].mean():.2f} seconds")
    print(f"Average Peak Memory: {results_df['peak_memory'].mean():.2f} MiB")

Starting processing of directory: /gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_07_sd_eps_3
Processing file 1/1000: simulated_data_2_seed_20250473_sdZ_07_sdEps_3.csv
Epoch 1/50, Train Loss: 9771.0983, Val Loss: 7673.2012
Epoch 2/50, Train Loss: 8865.9967, Val Loss: 6460.7056
Epoch 3/50, Train Loss: 6877.3752, Val Loss: 4539.5088
Epoch 4/50, Train Loss: 4757.3180, Val Loss: 2980.4238
Epoch 5/50, Train Loss: 3064.4672, Val Loss: 1770.1123
Epoch 6/50, Train Loss: 1817.8767, Val Loss: 944.9991
Epoch 7/50, Train Loss: 1040.9463, Val Loss: 524.5591
Epoch 8/50, Train Loss: 667.2166, Val Loss: 367.0117
Epoch 9/50, Train Loss: 540.7671, Val Loss: 331.4728
Epoch 10/50, Train Loss: 510.0153, Val Loss: 328.3423
Epoch 11/50, Train Loss: 504.8648, Val Loss: 328.2919
Epoch 12/50, Train Loss: 499.3863, Val Loss: 326.9067
Epoch 13/50, Train Loss: 497.7448, Val Loss: 325.3386
Epoch 14/50, Train Loss: 495.5692, Val Loss: 323.8005
Epoch 15/50, Train L

In [ ]:
import pandas as pd
import numpy as np

def calculate_metrics(csv_path, true_effect_111, true_effect_011):
    """
    Calculate performance metrics for causal effect estimates
    
    Args:
        csv_path: Path to results CSV file
        true_effect_111: True causal effect for [1,1,1] vs [0,0,0]
        true_effect_011: True causal effect for [0,1,1] vs [0,0,0]
    
    Returns:
        Dictionary containing all six metrics
    """
    # Read results
    df = pd.read_csv(csv_path)
    
    # Calculate metrics for effect_111
    mean_hat_111 = df['effect_111'].mean()
    rel_bias_111 = (mean_hat_111 - true_effect_111) / true_effect_111
    mcsd_111 = df['effect_111'].std(ddof=1)  # Using H-1 in denominator
    rmse_111 = np.sqrt(((df['effect_111'] - true_effect_111)**2).mean())
    
    # Calculate metrics for effect_011
    mean_hat_011 = df['effect_011'].mean()
    rel_bias_011 = (mean_hat_011 - true_effect_011) / true_effect_011
    mcsd_011 = df['effect_011'].std(ddof=1)  # Using H-1 in denominator
    rmse_011 = np.sqrt(((df['effect_011'] - true_effect_011)**2).mean())
    
    return {
        'effect_111': {
            'relative_bias': rel_bias_111,
            'mcsd': mcsd_111,
            'rmse': rmse_111
        },
        'effect_011': {
            'relative_bias': rel_bias_011,
            'mcsd': mcsd_011,
            'rmse': rmse_011
        }
    }

# Example usage
if __name__ == "__main__":
    # Replace with your actual CSV path and true effect values
    results_csv = "/gpfs/gibbs/project/guan_leying/ch2343/causal_time_predict/simulation_data_full/parallel_linear_sd_Z_07_sd_eps_3_results_RNN_modified.csv"
    
    # You'll need to provide these true effect values from your large simulation
    true_effect_111 = 5.0  # Replace with your actual value
    true_effect_011 = 3.0  # Replace with your actual value
    
    metrics = calculate_metrics(results_csv, 5.041, 3.021)
    
    # Print results
    print("Performance Metrics:")
    print("\nFor Effect [1,1,1] vs [0,0,0]:")
    print(f"Relative Bias: {metrics['effect_111']['relative_bias']:.4f}")
    print(f"Monte Carlo SD: {metrics['effect_111']['mcsd']:.4f}")
    print(f"RMSE: {metrics['effect_111']['rmse']:.4f}")
    
    print("\nFor Effect [0,1,1] vs [0,0,0]:")
    print(f"Relative Bias: {metrics['effect_011']['relative_bias']:.4f}")
    print(f"Monte Carlo SD: {metrics['effect_011']['mcsd']:.4f}")
    print(f"RMSE: {metrics['effect_011']['rmse']:.4f}")